#Objective: using the code format

In [10]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.2/36.2 MB 39.5 MB/s eta 0:00:00


In [26]:
import re, h5py, numpy as np, pandas as pd
from difflib import get_close_matches
import os, re, unicodedata, math
# 0) imports you might be missing (safe to repeat)
import torch, h5py
from typing import List
from difflib import get_close_matches
from torch import nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from rdkit import Chem
from rdkit.Chem import AllChem, Draw
import os, re, h5py, math, numpy as np, pandas as pd
from typing import List
from difflib import get_close_matches
from collections import Counter
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import GroupShuffleSplit

#Loading Embeddings

In [3]:
H5_PATH = "Seqs_list_total.h5"  # your H5 with per-sequence embeddings

# If heat_long isn't in memory, load it from disk:
try:
    heat_long
except NameError:
    heat_long = pd.read_csv("ipsita_heatmap_long.xlsx")

assert "Enzyme" in heat_long.columns, "heat_long must have an 'Enzyme' column."

# ---- helpers to enumerate H5 datasets ----
def collect_h5_datasets(h5_path):
    paths = []
    with h5py.File(h5_path, "r") as f:
        def visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                paths.append(name)
        f.visititems(visit)
    return paths

def norm_key(s: str) -> str:
    # Uppercase, alphanumeric only (robust matching)
    return re.sub(r"[^A-Za-z0-9]", "", str(s)).upper()

def last_token(code: str) -> str:
    # Take suffix after the last underscore (e.g., 'Lacto_A0A5C2C1U8' -> 'A0A5C2C1U8')
    return str(code).split("_")[-1]

# ---- scan H5 once and index by normalized base name ----
all_paths = collect_h5_datasets(H5_PATH)
if not all_paths:
    raise RuntimeError(f"No datasets found in {H5_PATH}.")

bases = [p.rsplit("/", 1)[-1] for p in all_paths]       # final path component only
norm_bases = [norm_key(b) for b in bases]
normbase_to_path = {}
for base, normb, full in zip(bases, norm_bases, all_paths):
    normbase_to_path.setdefault(normb, full)             # first occurrence wins

all_norm_bases = list(normbase_to_path.keys())

def pick_contains_match(tok_norm: str):
    hits = [b for b in all_norm_bases if tok_norm in b or b in tok_norm]
    if not hits: return None
    hits = sorted(hits, key=len)  # shortest base first (often the bare accession)
    return normbase_to_path[hits[0]]

def pick_fuzzy_match(tok_norm: str, cutoff=0.92):
    cand = get_close_matches(tok_norm, all_norm_bases, n=1, cutoff=cutoff)
    return (normbase_to_path[cand[0]] if cand else None)

def match_dataset_for_enzyme(enzyme_id: str) -> str | None:
    """
    Try multiple strategies:
      1) exact match on full normalized enzyme string;
      2) exact match on last-token (accession-like);
      3) contains fallback;
      4) fuzzy fallback.
    Return full H5 dataset path or None.
    """
    full_norm = norm_key(enzyme_id)
    tok_norm  = norm_key(last_token(enzyme_id))

    # 1) exact on full norm
    if full_norm in normbase_to_path:
        return normbase_to_path[full_norm]
    # 2) exact on last token
    if tok_norm in normbase_to_path:
        return normbase_to_path[tok_norm]
    # 3) contains
    p = pick_contains_match(tok_norm)
    if p: return p
    # 4) fuzzy
    return pick_fuzzy_match(tok_norm, cutoff=0.92)

def fetch_and_pool(h5_path, full_path):
    with h5py.File(h5_path, "r") as f:
        arr = np.array(f[full_path])
    # mean-pool per-residue to a single vector
    if arr.ndim == 2 and arr.shape[0] > 1:
        arr = arr.mean(axis=0)
    elif arr.ndim == 2 and arr.shape[0] == 1:
        arr = arr[0]
    elif arr.ndim > 2:
        arr = arr.reshape(arr.shape[-1])
    return arr.astype(np.float32)

# ---- do the matching for enzymes present in heat_long ----
enz_ids = sorted(heat_long["Enzyme"].astype(str).unique().tolist())
rows, unmatched = [], []
for eid in enz_ids:
    path = match_dataset_for_enzyme(eid)
    if path is None:
        unmatched.append(eid); continue
    try:
        vec = fetch_and_pool(H5_PATH, path)
        rows.append((eid, path, vec, int(vec.shape[-1])))
    except Exception as e:
        print(f"[Warn] failed to read {eid} at {path}: {e}")

if not rows:
    raise RuntimeError("No embeddings loaded—check H5 structure and enzyme naming.")

df_enz = pd.DataFrame(rows, columns=["Enzyme", "h5_path", "Embedding", "dim"])
E = np.stack(df_enz["Embedding"].values, axis=0).astype(np.float32)
d_prot = E.shape[1]
enzyme2idx = {eid: i for i, eid in enumerate(df_enz["Enzyme"].tolist())}

print(f"[OK] Embedding matrix: {E.shape} (d_prot={d_prot})")
print(f"[Match] {len(df_enz)} matched / {len(enz_ids)} enzymes in heat_long.")
if unmatched[:10]:
    print("[Unmatched] examples:", unmatched[:10])

# (Optional) save embeddings for reuse
np.save("enzyme_embeddings.npy", {row.Enzyme: row.Embedding for _, row in df_enz.iterrows()})
emb_cols = [f"enz_{i}" for i in range(d_prot)]
pd.concat(
    [df_enz[["Enzyme"]].reset_index(drop=True),
     pd.DataFrame(E, columns=emb_cols)],
    axis=1
).to_csv("enzyme_embeddings.csv", index=False)
print("Saved: enzyme_embeddings.npy (dict) and enzyme_embeddings.csv (flat table)")


[OK] Embedding matrix: (132, 1024) (d_prot=1024)
[Match] 132 matched / 135 enzymes in heat_long.
[Unmatched] examples: ['A0A1Y4QH48', 'Pencillin_amidase', 'nan']
Saved: enzyme_embeddings.npy (dict) and enzyme_embeddings.csv (flat table)


In [5]:
df_enz.head()

,Enzyme,h5_path,Embedding,dim
0,A0A069S9I0,cluster54_A0A069S9I0,"[-0.017257635, 0.04989724, 0.03538562, 0.00854...",1024
1,A0A078S2R5,cluster23_A0A078S2R5,"[-0.010787812, 0.05008991, 0.03887677, 0.00458...",1024
2,A0A096B1U0,cluster183_A0A096B1U0,"[-0.057331573, 0.025941156, 0.06507717, 0.0291...",1024
3,A0A096DGB7,cluster245_A0A096DGB7,"[-0.019923376, 0.024629124, 0.024821218, 0.018...",1024
4,A0A096KTH7,cluster10_A0A096KTH7,"[-0.045196902, 0.010763844, 0.041411456, 0.022...",1024


#Creating path to keep track of all possible pairs

Current Path

1. substrates_catalog_pairs.csv → the dictionary of all unique BA–Amine combinations (one row per pair with names/SMILES and a stable feat_idx).

2. "group_membership_with_featidx.csv" → the index that maps each ProductName to its dictionary entry via sub_key and feat_idx.

3. substrate_features_pairs.npz → fingeprint for each dictionary entry

In [14]:
ENUM_XLSX = "swap_enumeration_FINAL.xlsx"
if os.path.exists(ENUM_XLSX):
    enum_df = pd.read_excel(ENUM_XLSX)
elif os.path.exists(ENUM_CSV):
    enum_df = pd.read_csv(ENUM_CSV)
else:
    raise FileNotFoundError("swap_enumeration_FINAL.(xlsx/csv) not found.")

# columns we will use (make them exist if missing, kept robust)
for c in ["ProductName","Parent_BA_Name","Parent_BA_SMILES_Original",
          "Acid_SMILES_Used","Amine_Name","Amine_SMILES","Note"]:
    if c not in enum_df.columns:
        enum_df[c] = np.nan

# ---------- helpers ----------
def split_products(s: str):
    if pd.isna(s): return []
    parts = [p.strip() for p in str(s).split(";") if str(p).strip()]
    return parts if parts else []

def pick_ba_smiles(row):
    # prefer the original BA SMILES; fall back to the free-acid we actually amidated
    for c in ["Parent_BA_SMILES_Original","Acid_SMILES_Used"]:
        smi = str(row.get(c, "") or "").strip()
        if smi: return smi
    return ""

def morgan_fp(smi, nBits=1024, radius=2):
    if not smi: return np.zeros(nBits, dtype=np.float32)
    m = Chem.MolFromSmiles(str(smi))
    if m is None: return np.zeros(nBits, dtype=np.float32)
    bv = AllChem.GetMorganFingerprintAsBitVect(m, radius=radius, nBits=nBits)
    arr = np.zeros((nBits,), dtype=np.int8)
    Chem.DataStructs.ConvertToNumpyArray(bv, arr)
    return arr.astype(np.float32)

def name_hash_fp(name: str, nBits=256):
    # stable fallback so we never drop a pair for missing SMILES
    s = (str(name) or "").lower().strip()
    grams = set(); t = f"^{s}$"
    for i in range(max(0, len(t)-2)):
        grams.add(t[i:i+3])
    bits = np.zeros(nBits, dtype=np.float32)
    for g in grams:
        idx = (hash(g) % nBits + nBits) % nBits
        bits[idx] = 1.0
    return bits

# ---------- 1) membership: (ProductName, sub_key) ----------
rows = []
for _, r in enum_df.iterrows():
    pnames = split_products(r["ProductName"])
    pname  = pnames if pnames else [str(r["ProductName"]).strip()]
    ba_nm  = str(r["Parent_BA_Name"] or "").strip()
    am_nm  = str(r["Amine_Name"] or "").strip()
    if not ba_nm and not am_nm:
        continue
    sub_key = f"{ba_nm} || {am_nm}"
    for pn in pname:
        if pn:
            rows.append((pn, sub_key))
membership = pd.DataFrame(rows, columns=["ProductName","sub_key"]).drop_duplicates().reset_index(drop=True)
print(f"[membership] products={membership['ProductName'].nunique()}  rows={len(membership)}")

# ---------- 2) substrates_catalog_pairs: unique BA||Amine with SMILES ----------
rows = []
seen = set()
for _, r in enum_df.iterrows():
    ba_nm = str(r["Parent_BA_Name"] or "").strip()
    am_nm = str(r["Amine_Name"] or "").strip()
    if not ba_nm and not am_nm:
        continue
    sk = f"{ba_nm} || {am_nm}"
    if sk in seen:
        continue
    seen.add(sk)
    rows.append({
        "sub_key": sk,
        "parent_ba_name": ba_nm,
        "amine_name": am_nm,
        "ba_smiles": pick_ba_smiles(r),
        "amine_smiles": (str(r["Amine_SMILES"]) or "").strip(),
    })
substrates = pd.DataFrame(rows).reset_index(drop=True)

# Fill any missing SMILES with hashed-name fallbacks later (when featurizing)

# Assign stable indices
substrates["feat_idx"] = np.arange(len(substrates), dtype=int)
subkey2idx = dict(zip(substrates["sub_key"], substrates["feat_idx"]))
membership["feat_idx"] = membership["sub_key"].map(subkey2idx)

# ---------- 3) compute features ----------
# (a) Amine Morgan-1024  (b) BA Morgan-512 (smaller is fine for BA)
A_bits, B_bits = [], []
nA_smi = nA_hash = nB_smi = nB_hash = 0
for _, r in substrates.iterrows():
    a_smi = (r["amine_smiles"] or "").strip()
    b_smi = (r["ba_smiles"] or "").strip()

    if a_smi:
        A = morgan_fp(a_smi, nBits=1024, radius=2); nA_smi += 1
    else:
        A = name_hash_fp(r["amine_name"], nBits=1024);   nA_hash += 1

    if b_smi:
        B = morgan_fp(b_smi, nBits=512, radius=2);       nB_smi += 1
    else:
        B = name_hash_fp(r["parent_ba_name"], nBits=512); nB_hash += 1

    A_bits.append(A); B_bits.append(B)

A_bits = np.stack(A_bits, axis=0).astype(np.float32)   # [N_subkeys, 1024]
B_bits = np.stack(B_bits, axis=0).astype(np.float32)   # [N_subkeys,  512]

print(f"[features] amine: SMILES {nA_smi} | hash {nA_hash}  -> {A_bits.shape}")
print(f"[features] BA   : SMILES {nB_smi} | hash {nB_hash}  -> {B_bits.shape}")

# ---------- 4) save artifacts ----------
membership.to_csv("group_membership_with_featidx.csv", index=False)
substrates.to_csv("substrates_catalog_pairs.csv", index=False)
np.savez_compressed(
    "substrate_features_pairs.npz",
    A=A_bits, B=B_bits,
    sub_keys=substrates["sub_key"].values,
    parent_ba_names=substrates["parent_ba_name"].values,
    amine_names=substrates["amine_name"].values,
    feat_idx=substrates["feat_idx"].values,
)
print("Saved: group_membership_with_featidx.csv, substrates_catalog_pairs.csv, substrate_features_pairs.npz")


[membership] products=94  rows=651


[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerator
[17:08:51] DEPRECATION WARNING: please use MorganGenerat

[features] amine: SMILES 354 | hash 0  -> (354, 1024)
[features] BA   : SMILES 354 | hash 0  -> (354, 512)
Saved: group_membership_with_featidx.csv, substrates_catalog_pairs.csv, substrate_features_pairs.npz


Substrates: Accoutns for the bile acid + amine smiels that accoutn for the product

In [18]:
substrates.head()

,sub_key,parent_ba_name,amine_name,ba_smiles,amine_smiles,feat_idx
0,3a7a12k-Gly || L-Alanine,3a7a12k-Gly,L-Alanine,C[C@H](CCC(=O)NCC(=O)O)[C@H]1CC[C@H]2[C@@H]3[C...,C[C@@H](C(=O)O)N,0
1,3a7a12k-Tau || L-Alanine,3a7a12k-Tau,L-Alanine,C[C@H](CCC(=O)NCCS(=O)(=O)O)[C@H]1CC[C@H]2[C@@...,C[C@@H](C(=O)O)N,1
2,3a7a12k-Gly || L-Lysine,3a7a12k-Gly,L-Lysine,C[C@H](CCC(=O)NCC(=O)O)[C@H]1CC[C@H]2[C@@H]3[C...,C(CCN)C[C@@H](C(=O)O)N,2
3,3a7a12k-Tau || L-Lysine,3a7a12k-Tau,L-Lysine,C[C@H](CCC(=O)NCCS(=O)(=O)O)[C@H]1CC[C@H]2[C@@...,C(CCN)C[C@@H](C(=O)O)N,3
4,3a12k-Gly || L-Serine,3a12k-Gly,L-Serine,C[C@H](CCC(=O)NCC(=O)O)[C@H]1CC[C@H]2[C@@H]3CC...,C([C@@H](C(=O)O)N)O,4


Index table that goe back to the substrate table

In [20]:
membership.head()

,ProductName,sub_key,feat_idx
0,"3a,7a,12k_alanine_20366",3a7a12k-Gly || L-Alanine,0
1,"3a,7a,12k_alanine_20366",3a7a12k-Tau || L-Alanine,1
2,"3a,7a,12k_lysine_23168",3a7a12k-Gly || L-Lysine,2
3,"3a,7a,12k_lysine_23168",3a7a12k-Tau || L-Lysine,3
4,3a12k_serine_27652,3a12k-Gly || L-Serine,4


#Multiple Instance Learning Model- Which bile acid would be the best for the BA+Amine pair?

Inputs:
E = enzyme embedding A = amine fingerprint  B=  bile-acid fingerprints

Scorer (MLP): Determines which bile acid is most compatible with the specific amine

Mixer (attention): computes weights over the BA set

In [ ]:
CFG = dict(
    seed=1337,
    device="cuda" if torch.cuda.is_available() else "cpu",
    label_log1p=True,           # ✅ turn this on for LC-MS scales
    batch_size=256,
    epochs=40,
    lr=3e-4,
    weight_decay=1e-4,
    grad_clip=1.0,
    lambda_entropy=1e-3,        # added to loss => encourages sparse attention (low entropy)
    val_frac=0.15,
    test_frac=0.15,
    # Early stop + checkpoint
    patience=8,
    min_delta=1e-4,
    ckpt_path="best_mil.pt",
    # Inspector
    topk_show=5,
)

# ------------- utils -------------
def set_seeds(seed:int):
    import random
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True; torch.backends.cudnn.benchmark = False

def rmse(a,b): a,b=np.asarray(a),np.asarray(b); return float(np.sqrt(np.mean((a-b)**2)))
def mae(a,b):  a,b=np.asarray(a),np.asarray(b); return float(np.mean(np.abs(a-b)))
def pearsonr(a,b):
    a,b=np.asarray(a),np.asarray(b)
    if a.std()==0 or b.std()==0: return 0.0
    return float(np.corrcoef(a,b)[0,1])

def norm_key(s): return re.sub(r"[^A-Za-z0-9]","",str(s)).upper()
def last_token(code:str)->str: return str(code).split("_")[-1]

# ------------- labels (heatmaps → long) -------------
def load_labels() -> pd.DataFrame:
    df_am = pd.read_csv("NEW_Stage2_BAs_amines_for_heatmap_manual.csv")
    df_sb = pd.read_csv("NEW_Stage2_BAs_subs_for_heatmap_manual.csv")
    meta = [c for c in ['Code','Replicate','filename'] if c in df_am.columns]

    def to_long(df):
        long = df.melt(id_vars=meta, var_name="ProductName", value_name="y")
        long["y"] = pd.to_numeric(long["y"], errors="coerce")
        return long.dropna(subset=["y"]).rename(columns={"Code":"enzyme_id","ProductName":"group_id"})

    long = pd.concat([to_long(df_am), to_long(df_sb)], ignore_index=True)
    agg = long.groupby(["enzyme_id","group_id"], as_index=False)["y"].mean()
    return agg

labels_long = load_labels()
print(f"[labels] rows={len(labels_long)} enzymes={labels_long['enzyme_id'].nunique()} groups={labels_long['group_id'].nunique()}")

# ------------- membership & pair features -------------
mem = pd.read_csv("group_membership_with_featidx.csv")      # ProductName, sub_key, feat_idx
sub = pd.read_csv("substrates_catalog_pairs.csv")           # feat_idx, parent_ba_name, amine_name, ba_smiles, amine_smiles

pairs_npz = np.load("substrate_features_pairs.npz", allow_pickle=True)
A = pairs_npz["A"].astype(np.float32)                       # [N_subkeys, 1024]  (amine features)
B = pairs_npz["B"].astype(np.float32)                       # [N_subkeys,  512]  (bile acid features)
feat_idx = pairs_npz["feat_idx"].astype(int)                # [N_subkeys]
idx2row  = {int(fi):i for i,fi in enumerate(feat_idx.tolist())}

# group_id → list of (row_idx into A/B)
group_to_rows = {}
for gid, grp in mem.groupby("ProductName"):
    rows = []
    for fi in grp["feat_idx"].dropna().astype(int).tolist():
        if fi in idx2row: rows.append(idx2row[fi])
    group_to_rows[gid] = sorted(set(rows))
print(f"[groups] {len(group_to_rows)} groups with membership")

# ------------- amine vec per group (optional shortcut) -------------
# We keep a "group amine vector" by averaging amine FPs of the group's pairs.
def amine_vec_for_group(gid: str) -> np.ndarray | None:
    rows = group_to_rows.get(gid, [])
    if not rows: return None
    return A[rows].mean(axis=0)

# ------------- enzyme embeddings (H5) -------------
def collect_h5_datasets(h5_path):
    paths=[]
    with h5py.File(h5_path,"r") as f:
        def visit(n,o):
            if isinstance(o, h5py.Dataset): paths.append(n)
        f.visititems(visit)
    return paths

def load_enzyme_embeddings(h5_path: str, enzyme_ids: List[str]):
    all_paths = collect_h5_datasets(h5_path)
    bases = [p.rsplit("/",1)[-1] for p in all_paths]
    norm_map = {norm_key(b):p for b,p in zip(bases, all_paths)}
    id2path = {}
    for eid in sorted(set(enzyme_ids)):
        tok = norm_key(last_token(eid))
        if tok in norm_map: id2path[eid]=norm_map[tok]; continue
        cand = [k for k in norm_map if (tok in k or k in tok)]
        if cand: id2path[eid]=norm_map[sorted(cand,key=len)[0]]; continue
        fm = get_close_matches(tok, list(norm_map.keys()), n=1, cutoff=0.92)
        if fm: id2path[eid]=norm_map[fm[0]]

    rows=[]
    with h5py.File(h5_path,"r") as f:
        for eid, pth in id2path.items():
            arr = np.array(f[pth])
            vec = arr.mean(axis=0).astype(np.float32) if arr.ndim==2 else arr.astype(np.float32)
            rows.append((eid, pth, vec, int(vec.shape[-1])))
    df_enz = pd.DataFrame(rows, columns=["enzyme_id","h5_path","Embedding","dim"])
    E = np.stack(df_enz["Embedding"].values, axis=0).astype(np.float32)
    enz2idx = {eid:i for i,eid in enumerate(df_enz["enzyme_id"].tolist())}
    print(f"[enz] loaded {len(df_enz)} embeddings (d_prot={E.shape[1]}); "
          f"missing={len(set(enzyme_ids)-set(enz2idx))}")
    return df_enz, enz2idx, E

df_enz, enzyme2idx, E_mat = load_enzyme_embeddings("Seqs_list_total.h5", labels_long["enzyme_id"].tolist())
d_prot = int(E_mat.shape[1])

# ------------- build examples -------------
def build_examples(labels: pd.DataFrame) -> pd.DataFrame:
    rows=[]
    skipped=0
    for _, r in labels.iterrows():
        eid = r["enzyme_id"]; gid = r["group_id"]; y = float(r["y"])
        if eid not in enzyme2idx: skipped+=1; continue
        row_ids = group_to_rows.get(gid, [])
        if not row_ids: skipped+=1; continue
        Avec = amine_vec_for_group(gid)
        if Avec is None: skipped+=1; continue
        rows.append({
            "enzyme_id": eid,
            "group_id": gid,
            "y": math.log1p(y) if CFG["label_log1p"] else y,
            "enz_idx": enzyme2idx[eid],
            "amine_vec": Avec,       # [1024]
            "row_ids": row_ids,      # list of indices into A/B
        })
    df = pd.DataFrame(rows)
    print(f"[examples] {len(df)} built | skipped {skipped}")
    return df

examples = build_examples(labels_long)

# ------------- dataset -------------
class PoolDataset(Dataset):
    def __init__(self, df, E_mat, A_mat, B_mat):
        self.df = df.reset_index(drop=True)
        self.E  = E_mat; self.A = A_mat; self.B = B_mat
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        r = self.df.iloc[i]
        Evec = self.E[int(r["enz_idx"])].astype(np.float32)
        Avec = np.asarray(r["amine_vec"], dtype=np.float32)
        idxs = list(map(int, r["row_ids"]))
        A_k  = self.A[idxs].astype(np.float32)   # [k, 1024] (not used by current scorer)
        B_k  = self.B[idxs].astype(np.float32)   # [k, 512]
        y    = float(r["y"])
        return (torch.from_numpy(Evec), torch.from_numpy(Avec),
                torch.from_numpy(A_k),  torch.from_numpy(B_k),
                torch.tensor(y, dtype=torch.float32))

def collate(batch):
    E,A,Ak,Bk,Y = zip(*batch)
    E = torch.stack(E,0); A = torch.stack(A,0); Y = torch.stack(Y,0)
    # keep variable-length lists for A_k / B_k
    return E, A, list(Ak), list(Bk), Y

# ------------- grouped splits by enzyme (no leakage) -------------
# ------------- grouped splits by enzyme (no leakage) -------------
from sklearn.model_selection import GroupShuffleSplit

def group_splits_by_enzyme(df, val_frac=0.15, test_frac=0.15, seed=1337):
    """
    Return (train_df, val_df, test_df) with no enzyme leakage across splits.
    IMPORTANT: compute val split on the *base* train set, then materialize
    train/val from that base (avoid indexer mismatch).
    """
    # Outer split: hold out test by enzyme
    groups = df["enzyme_id"].values
    gss_outer = GroupShuffleSplit(n_splits=1, test_size=test_frac, random_state=seed)
    tr_idx, te_idx = next(gss_outer.split(df, groups=groups))
    base_train = df.iloc[tr_idx].reset_index(drop=True)   # <-- keep a BASE copy
    test_pool  = df.iloc[te_idx].reset_index(drop=True)

    # Inner split: carve validation from BASE train by enzyme
    gss_inner = GroupShuffleSplit(
        n_splits=1,
        test_size=val_frac / (1.0 - test_frac),
        random_state=seed
    )
    tr2_idx, va_idx = next(gss_inner.split(base_train, groups=base_train["enzyme_id"].values))
    train_pool = base_train.iloc[tr2_idx].reset_index(drop=True)  # <-- use base_train
    val_pool   = base_train.iloc[va_idx].reset_index(drop=True)   # <-- use base_train

    # Sanity: ensure disjoint enzyme sets
    assert set(train_pool["enzyme_id"]).isdisjoint(val_pool["enzyme_id"])
    assert set(train_pool["enzyme_id"]).isdisjoint(test_pool["enzyme_id"])
    assert set(val_pool["enzyme_id"]).isdisjoint(test_pool["enzyme_id"])

    print(f"[split] enzymes train={train_pool['enzyme_id'].nunique()} "
          f"val={val_pool['enzyme_id'].nunique()} test={test_pool['enzyme_id'].nunique()}")
    print(f"[split] rows    train={len(train_pool)} val={len(val_pool)} test={len(test_pool)}")
    return train_pool, val_pool, test_pool

# Rebuild datasets/loaders from split DataFrames
train_pool, val_pool, test_pool = group_splits_by_enzyme(
    examples, val_frac=CFG["val_frac"], test_frac=CFG["test_frac"], seed=CFG["seed"]
)

def make_subset_dataset(pool_df):
    return PoolDataset(pool_df, E_mat, A, B)

train_ds = make_subset_dataset(train_pool)
val_ds   = make_subset_dataset(val_pool)
test_ds  = make_subset_dataset(test_pool)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate)


# datasets/loaders
def make_subset_dataset(pool_df):
    return PoolDataset(pool_df, E_mat, A, B)

train_ds = make_subset_dataset(train_pool)
val_ds   = make_subset_dataset(val_pool)
test_ds  = make_subset_dataset(test_pool)

train_loader = DataLoader(train_ds, batch_size=CFG["batch_size"], shuffle=True,  collate_fn=collate)
val_loader   = DataLoader(val_ds,   batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate)
test_loader  = DataLoader(test_ds,  batch_size=CFG["batch_size"], shuffle=False, collate_fn=collate)

print(f"[split] rows train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")

# ------------- model (fixed) -------------
class PairScorer(nn.Module):
    """Score each BA candidate: f_theta(E, A, B_k) -> y_{e,(a,Bk)} >= 0"""
    def __init__(self, d_prot, d_am, d_ba, hidden=512, depth=2, dropout=0.1):
        super().__init__()
        dims = [d_prot + d_am + d_ba] + [hidden]*(depth-1) + [1]
        layers=[]
        for i in range(len(dims)-2):
            layers += [nn.Linear(dims[i], dims[i+1]), nn.ReLU(), nn.Dropout(dropout)]
        layers += [nn.Linear(dims[-2], dims[-1])]
        self.mlp = nn.Sequential(*layers)
        self.softplus = nn.Softplus()

    def forward_one(self, E, A, Bk):
        """
        E:  [d_prot] (1D)
        A:  [d_am]   (1D)
        Bk: [k, d_ba]
        returns: [k] nonnegative scores
        """
        if Bk.dim() == 1:
            Bk = Bk.unsqueeze(0)  # [1, d_ba]
        k = Bk.size(0)
        Eexp = E.unsqueeze(0).expand(k, -1)  # [k, d_prot]
        Aexp = A.unsqueeze(0).expand(k, -1)  # [k, d_am]
        x = torch.cat([Eexp, Aexp, Bk], dim=-1)  # [k, d_prot+d_am+d_ba]
        return self.softplus(self.mlp(x)).squeeze(-1)  # [k]

    def forward(self, E, A, Bk):
        return self.forward_one(E, A, Bk)

class Mixer(nn.Module):
    """Enzyme-conditioned attention over BA candidates."""
    def __init__(self, d_prot, d_am, d_ba, d_att=128):
        super().__init__()
        self.q = nn.Sequential(nn.Linear(d_prot + d_am, d_att), nn.Tanh())
        self.k = nn.Linear(d_ba, d_att, bias=False)

    def forward_one(self, E, A, Bk, lambda_entropy=1e-3):
        if Bk.dim() == 1:
            Bk = Bk.unsqueeze(0)
        q = self.q(torch.cat([E, A], dim=-1))  # [d_att]
        K = self.k(Bk)                         # [k, d_att]
        w = torch.softmax(K @ q, dim=0)        # [k]
        ent = -(w * (w.clamp_min(1e-8).log())).sum() * lambda_entropy
        return w, ent

    def forward(self, E, A, Bk, lambda_entropy=1e-3):
        return self.forward_one(E, A, Bk, lambda_entropy=lambda_entropy)

class PooledModel(nn.Module):
    def __init__(self, d_prot, d_am, d_ba):
        super().__init__()
        self.scorer = PairScorer(d_prot, d_am, d_ba, hidden=512, depth=2, dropout=0.1)
        self.mixer  = Mixer(d_prot, d_am, d_ba, d_att=128)

    def forward(self, E, A, Ak_list, Bk_list, lambda_entropy=1e-3):
        B = E.size(0)
        Yhat = []
        total_ent = E.new_tensor(0.0)
        for b in range(B):
            yk = self.scorer(E[b], A[b], Bk_list[b])                     # [k]
            w, ent = self.mixer(E[b], A[b], Bk_list[b], lambda_entropy)  # [k], scalar
            Yhat.append((w * yk).sum())
            total_ent = total_ent + ent
        return torch.stack(Yhat, 0), total_ent / max(1, B)

# ------------- early stopping + checkpoint -------------
class EarlyStopper:
    def __init__(self, patience=8, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best = float("inf")
        self.bad = 0
    def step(self, metric: float) -> bool:
        """Returns True if improved."""
        if metric < (self.best - self.min_delta):
            self.best = metric
            self.bad = 0
            return True
        self.bad += 1
        return False
    def should_stop(self) -> bool:
        return self.bad >= self.patience

def save_ckpt(path, model, opt, epoch, val_metrics, cfg):
    os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
    torch.save({
        "model": model.state_dict(),
        "opt": opt.state_dict(),
        "epoch": epoch,
        "val": val_metrics,
        "cfg": cfg,
    }, path)

def load_ckpt(path, model, opt=None, map_location=None):
    d = torch.load(path, map_location=map_location or CFG["device"])
    model.load_state_dict(d["model"])
    if opt is not None and "opt" in d:
        opt.load_state_dict(d["opt"])
    return d

# ------------- train/eval loops -------------
def run_epoch(loader, train_mode: bool, model=None, opt=None):
    model.train(train_mode)
    y_true, y_pred = [], []
    tot_loss = tot_ent = 0.0
    for E,A,Ak_list,Bk_list,Y in loader:
        E = E.to(CFG["device"]).float()
        A = A.to(CFG["device"]).float()
        Ak_list = [x.to(CFG["device"]).float() for x in Ak_list]  # (not used here, kept for symmetry)
        Bk_list = [x.to(CFG["device"]).float() for x in Bk_list]
        Y = Y.to(CFG["device"]).float()

        Yhat, ent = model(E, A, Ak_list, Bk_list, lambda_entropy=CFG["lambda_entropy"])
        loss_data = F.mse_loss(Yhat, Y)
        loss = loss_data + ent

        if train_mode:
            opt.zero_grad(); loss.backward()
            if CFG["grad_clip"]: nn.utils.clip_grad_norm_(model.parameters(), CFG["grad_clip"])
            opt.step()
        y_true.extend(Y.detach().cpu().numpy()); y_pred.extend(Yhat.detach().cpu().numpy())
        tot_loss += float(loss_data.item()) * len(Y); tot_ent += float(ent.item()) * len(Y)

    n = len(y_true)
    y_true = np.array(y_true); y_pred = np.array(y_pred)
    return dict(
        loss=tot_loss/max(1,n), ent=tot_ent/max(1,n),
        rmse=rmse(y_true, y_pred), mae=mae(y_true, y_pred), r=pearsonr(y_true, y_pred),
    )

# ------------- build + train -------------
set_seeds(CFG["seed"])
d_prot = int(E_mat.shape[1]); d_am = A.shape[1]; d_ba = B.shape[1]
model = PooledModel(d_prot, d_am, d_ba).to(CFG["device"])
opt = torch.optim.AdamW(model.parameters(), lr=CFG["lr"], weight_decay=CFG["weight_decay"])
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=CFG["epochs"])

early = EarlyStopper(patience=CFG["patience"], min_delta=CFG["min_delta"])
best = {"rmse": float("inf")}
best_ep = -1

for ep in range(1, CFG["epochs"]+1):
    tr = run_epoch(train_loader, True,  model, opt)
    va = run_epoch(val_loader,   False, model, None)
    sched.step()

    print(f"[{ep:03d}] train RMSE {tr['rmse']:.3f} MAE {tr['mae']:.3f} r {tr['r']:.3f} | "
          f"val RMSE {va['rmse']:.3f} MAE {va['mae']:.3f} r {va['r']:.3f}  H(ent) {tr['ent']:.4f}")

    if early.step(va["rmse"]):
        best = va.copy(); best_ep = ep
        save_ckpt(CFG["ckpt_path"], model, opt, ep, va, CFG)

    if early.should_stop():
        print(f"[early-stop] no improvement for {CFG['patience']} epochs. Best val RMSE={best['rmse']:.3f} @ epoch {best_ep}.")
        break

# load best and evaluate test
_ = load_ckpt(CFG["ckpt_path"], model, opt=None, map_location=CFG["device"])
te = run_epoch(test_loader, False, model, None)
print(f"[TEST] RMSE {te['rmse']:.3f} MAE {te['mae']:.3f} r {te['r']:.3f} (best val RMSE {best['rmse']:.3f} @ ep {best_ep})")

# ------------- Top-k attention inspector -------------
# fast lookups to map row_idx -> metadata
feat_idx_arr = feat_idx  # aligned with A/B rows
sub_by_feat  = sub.set_index("feat_idx", drop=True)

def rowidx_to_info(row_idx: int) -> dict:
    fi = int(feat_idx_arr[row_idx])
    info = sub_by_feat.loc[fi]
    return dict(
        feat_idx=fi,
        parent_ba_name=str(info["parent_ba_name"]),
        amine_name=str(info["amine_name"]),
        ba_smiles=str(info.get("ba_smiles", "")),
        amine_smiles=str(info.get("amine_smiles","")),
    )

@torch.no_grad()
def topk_attention_for(model, enzyme_id: str, group_id: str, topk: int = 5):
    """Return a DataFrame of the top-k contributors (by attention * score) for (enzyme, product)."""
    if enzyme_id not in enzyme2idx:
        raise KeyError(f"enzyme_id '{enzyme_id}' not in embeddings.")
    if group_id not in group_to_rows or len(group_to_rows[group_id]) == 0:
        raise KeyError(f"group_id '{group_id}' has no instances.")

    Evec = torch.from_numpy(E_mat[enzyme2idx[enzyme_id]]).float().to(CFG["device"])
    Avec_np = amine_vec_for_group(group_id)
    if Avec_np is None:
        raise KeyError(f"group_id '{group_id}' has no amine vector.")
    Avec = torch.from_numpy(Avec_np).float().to(CFG["device"])

    row_ids = group_to_rows[group_id]
    Bk = torch.from_numpy(B[row_ids]).float().to(CFG["device"])

    model.eval()
    yk = model.scorer(Evec, Avec, Bk)                         # [k]
    w, _ = model.mixer(Evec, Avec, Bk, lambda_entropy=0.0)    # [k]
    contrib = (w * yk).detach().cpu().numpy()
    order = contrib.argsort()[::-1]

    recs = []
    for rank, j in enumerate(order[:topk], 1):
        info = rowidx_to_info(row_ids[j])
        recs.append({
            "rank": rank,
            "feat_idx": info["feat_idx"],
            "parent_ba_name": info["parent_ba_name"],
            "amine_name": info["amine_name"],
            "weight": float(w[j].detach().cpu()),
            "score": float(yk[j].detach().cpu()),
            "weight_x_score": float(contrib[j]),
        })
    return pd.DataFrame(recs)

def sanity_print_topk(model, df_subset: pd.DataFrame, topk: int = 5, n_examples: int = 5, seed: int = 1337):
    """Pretty-print top-k contributors for a few (enzyme, product) examples."""
    rng = np.random.default_rng(seed)
    if len(df_subset) == 0:
        print("[topk] empty subset"); return
    pick_idx = rng.choice(len(df_subset), size=min(n_examples, len(df_subset)), replace=False)
    for i in pick_idx:
        r = df_subset.iloc[int(i)]
        eid, gid, y = r["enzyme_id"], r["group_id"], float(r["y"])
        print(f"\n=== {eid} | {gid} | y={y:.3f} ===")
        try:
            df_top = topk_attention_for(model, eid, gid, topk=topk)
            print(df_top.to_string(index=False))
        except Exception as ex:
            print(f"(no top-k available) {ex}")

# Inspect a few validation/test examples
sanity_print_topk(model, val_pool,  topk=CFG["topk_show"], n_examples=5, seed=CFG["seed"])
sanity_print_topk(model, test_pool, topk=CFG["topk_show"], n_examples=5, seed=CFG["seed"])

[labels] rows=12596 enzymes=134 groups=94
[groups] 94 groups with membership
[enz] loaded 132 embeddings (d_prot=1024); missing=2
[examples] 12408 built | skipped 188
[split] enzymes train=92 val=20 test=20
[split] rows    train=8648 val=1880 test=1880
[split] rows train=8648 val=1880 test=1880
[001] train RMSE 6.220 MAE 5.112 r 0.057 | val RMSE 5.745 MAE 5.330 r 0.300  H(ent) 0.0008
[002] train RMSE 5.346 MAE 4.863 r 0.483 | val RMSE 4.925 MAE 4.345 r 0.576  H(ent) 0.0012
[003] train RMSE 4.684 MAE 4.003 r 0.613 | val RMSE 4.592 MAE 3.743 r 0.619  H(ent) 0.0007
[004] train RMSE 4.426 MAE 3.636 r 0.661 | val RMSE 4.573 MAE 3.727 r 0.643  H(ent) 0.0007
[005] train RMSE 4.274 MAE 3.458 r 0.688 | val RMSE 4.386 MAE 3.417 r 0.663  H(ent) 0.0008
[006] train RMSE 4.215 MAE 3.327 r 0.698 | val RMSE 4.349 MAE 3.411 r 0.669  H(ent) 0.0008
[007] train RMSE 4.164 MAE 3.277 r 0.707 | val RMSE 4.357 MAE 3.466 r 0.669  H(ent) 0.0007
[008] train RMSE 4.128 MAE 3.250 r 0.713 | val RMSE 4.339 MAE 3.393